In [ ]:
from pathlib import Path

import requests
import pandas as pd
import io
import os
import json
import time

# ==============================================================================
# CONFIGURATION FOR MULTIPLE DATASETS
# ==============================================================================
TARGET_DATASETS = [
    {
        "file_name": "OECD_RD_GDP_2000_2025.csv",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "sdmx_query_key": "all",  # Small dataset, safe to download "all" and filter via Pandas
        "filters": {
            "UNIT_MEASURE": "PT_B1GQ",
            "MEASURE": "G"
        }
    },
    {
        "file_name": "OECD_Inflation_CPI_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PRICES@DF_PRICES_ALL",
        "sdmx_query_key": ".A.N.CPI.._T.N.GY",
        "filters": {} 
    },
    {
        "file_name": "OECD_Unemployment_Rate_2000_2025.csv",
        "agency_id": "OECD.ELS.SAE",
        "dataflow_id": "DSD_LFS@DF_LFS_INDIC",
        "sdmx_query_key": "all",  
        "filters": {
            "MEASURE": "UNE_RATE",    # Unemployment Rate
            "UNIT_MEASURE": "PT_LF_SUB", # % of Labour Force
            "SEX": "_T",              # Total (Male + Female)
            "AGE": "_T"               # Total Age
        }
    },
    {
        "file_name": "OECD_Productivity_Variation_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PDB@DF_PDB_LV",
        "sdmx_query_key": "all",
        "filters": {
            "MEASURE": "GDPHRS",
            "UNIT_MEASURE": "USD",   # or "EUR" / "CNCY"
            "PRICE_BASE": "V",       # "V" = current prices, "L" = constant/chained prices
        }
    },
    {
        "file_name": "OECD_Tertiary_Education_2000_2025.csv",
        "agency_id": "OECD.EDU.IMEP",
        "dataflow_id": "DSD_EAG_LSO_EA@DF_LSO_NEAC_DISTR_EA",
        "sdmx_query_key": "._T.Y25T64.ISCED11A_5T8..........OBS...A",  
        "filters": {} # Left empty because the API query key handles all the filtering on the server
    },
    
    {   "file_name": "OECD_GDP_Growth_2000_2025.csv",
        "agency_id": "OECD.SDD.NAD",
        "dataflow_id": "DSD_NAMAIN1@DF_QNA_EXPENDITURE_GROWTH_OECD",
        "sdmx_query_key": "A.Y..S1.S1.B1GQ._Z._Z._Z.PC.L.GY.T0102",
        "filters": {}  # No Pandas-side filtering needed; server-side key is already precise
    }
]

START_PERIOD = "2000"
END_PERIOD = "2025"
VERSION = "all" # Uses the latest dataset version
PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)"
}

# ==============================================================================
# BATCH EXTRACTION PROCESS
# ==============================================================================

for dataset in TARGET_DATASETS:
    print(f"Processing: {dataset['file_name']}...")
    
    # --------------------------------------------------------------------------
    # FILE EXISTENCE CHECK (SKIP IF ALREADY DOWNLOADED)
    # --------------------------------------------------------------------------
    output_path = DATA_RAW_DIR / dataset['file_name']
    if output_path.exists():
        print(f"  -> File '{dataset['file_name']}' already exists. Skipping download.\n")
        continue

    cwd_path = Path(dataset['file_name'])
    if output_path.exists() or cwd_path.exists():
        print(f"  -> File already exists (either {output_path} or {cwd_path}). Skipping download.\n")
        continue

    # Build the specific URL using the server-side query key to prevent memory overload
    query_key = dataset.get('sdmx_query_key', 'all')
    url = f"https://sdmx.oecd.org/public/rest/data/{dataset['agency_id']},{dataset['dataflow_id']},{VERSION}/{query_key}"
    
    params = {
        "startPeriod": START_PERIOD,
        "endPeriod": END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels"
    }

    # Retry with backoff if rate-limited
    max_retries = 4
    for attempt in range(max_retries):
        response = requests.get(url, params=params, headers=headers)
        if response.status_code == 403:
            wait = 10 * (attempt + 1)
            print(f"  -> Rate-limited (403). Waiting {wait}s before retry ({attempt + 1}/{max_retries})...")
            time.sleep(wait)
            continue
        break
    
    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code == 200:
        csv_stream = io.StringIO(response.text)
        df = pd.read_csv(csv_stream, low_memory=False)
        
        if 'OBS_VALUE' not in df.columns:
            print(f"  -> Error: No valid data returned by the API for {dataset['file_name']}.")
            continue
            
        df_clean = df.dropna(subset=['OBS_VALUE']).copy()
        
        # ── NEW: normalize OBS_VALUE by UNIT_MULT before any filtering ──────────
        if 'UNIT_MULT' in df_clean.columns:
            df_clean['UNIT_MULT'] = pd.to_numeric(df_clean['UNIT_MULT'], errors='coerce').fillna(0)
            df_clean['OBS_VALUE'] = df_clean['OBS_VALUE'] * (10 ** df_clean['UNIT_MULT'])
            print(f"  -> UNIT_MULT normalization applied (unique multipliers found: "
                f"{sorted(df_clean['UNIT_MULT'].unique().tolist())})")
        
        # Apply local Pandas filters only if they are defined in the dictionary
        for column_name, filter_value in dataset.get('filters', {}).items():
            if column_name in df_clean.columns:
                df_clean = df_clean[df_clean[column_name] == filter_value]
            else:
                print(f"  -> Warning: Column '{column_name}' not found.")
        
        if df_clean.empty:
            print(f"  -> Error: Dataset is empty after applying filters.")
            
            debug_dict = {}
            for col in df.columns:
                if col not in ['OBS_VALUE', 'TIME_PERIOD', 'REF_AREA']:
                    debug_dict[col] = df[col].dropna().unique().tolist()
            
            with open("DEBUG_OECD_CODES.txt", "w") as f:
                f.write(json.dumps(debug_dict, indent=4))
                
            print("  -> Debug file generated: 'DEBUG_OECD_CODES.txt'. Please check the exact codes.\n")
            continue
        
        target_columns = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
        
        if all(col in df_clean.columns for col in target_columns):
            df_tidy = df_clean[target_columns].rename(columns={
                'REF_AREA': 'Country',
                'TIME_PERIOD': 'Year',
                'OBS_VALUE': 'Value'
            })
            
            output_path = DATA_RAW_DIR / dataset['file_name']
            df_tidy = df_tidy.reset_index(drop=True)
            df_tidy.to_csv(output_path, index=False)
            
            print(f"  -> Success! Saved {len(df_tidy)} rows to {output_path}.\n")
        else:
            print(f"  -> Structural error: Missing SDMX columns in {dataset['file_name']}.\n")
    else:
        print(f"  -> HTTP Error {response.status_code} for {dataset['file_name']}.\n")

Processing: OECD_RD_GDP_2000_2025.csv...
  -> File 'OECD_RD_GDP_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Inflation_CPI_2000_2025.csv...
  -> File 'OECD_Inflation_CPI_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Unemployment_Rate_2000_2025.csv...
  -> File 'OECD_Unemployment_Rate_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Productivity_Variation_2000_2025.csv...
  -> Rate-limited (403). Waiting 10s before retry (1/4)...


In [ ]:
import itertools
from pathlib import Path
import pandas as pd
import requests
import country_converter as coco
import logging
logging.getLogger("country_converter").setLevel(logging.ERROR)

PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

EUROSTAT_BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
EUROSTAT_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)",
}

EUROSTAT_DATASETS = [
    {
        "file_name": "Eurostat_Public_Debt_GDP_2000_2025.csv",
        "dataset_code": "gov_10dd_edpt1",
        "params": {
            "format": "JSON",
            "lang": "EN",
            "sector": "S13",
            "unit": "PC_GDP",
            "na_item": "GD",
            "startPeriod": 2000,
            "endPeriod": 2025,
        },
        "keep_columns": ["geo", "time", "OBS_VALUE"],
        "rename_map": {"geo": "Country", "time": "Year", "OBS_VALUE": "Value"},
    },
]

WORLD_BANK_BASE = "https://api.worldbank.org/v2"
WORLD_BANK_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)",
}

EUROSTAT_GEO_OVERRIDES = {"EL": "GR", "UK": "GB"}

def convert_eurostat_geo_to_iso3(df_tidy: pd.DataFrame) -> pd.DataFrame:
    df_tidy = df_tidy.copy()
    standardized = df_tidy["Country"].replace(EUROSTAT_GEO_OVERRIDES)
    df_tidy["Country"] = coco.convert(
        names=standardized.tolist(), src="ISO2", to="ISO3", not_found=None
    )
    dropped = df_tidy["Country"].isna().sum()
    if dropped:
        print(f"  -> Dropped {dropped} rows with unmapped/aggregate geo codes.")
    return df_tidy.dropna(subset=["Country"]).reset_index(drop=True)


def get_world_bank_country_codes() -> set[str]:
    """Return the ISO3 codes for World Bank country rows, excluding aggregates."""
    response = requests.get(
        f"{WORLD_BANK_BASE}/country",
        params={"format": "json", "per_page": 400},
        headers=WORLD_BANK_HEADERS,
        timeout=120,
    )
    response.raise_for_status()

    payload = response.json()
    if not isinstance(payload, list) or len(payload) < 2:
        raise ValueError("Unexpected World Bank country metadata response format.")

    country_codes = set()
    for item in payload[1]:
        if not item:
            continue
        region = item.get("region", {})
        country_code = item.get("id")
        if region.get("id") != "NA" and country_code and len(country_code) == 3:
            country_codes.add(country_code)

    return country_codes

WORLD_BANK_DATASETS = [
    {
        "file_name": "Global_Energy_Dependency_0_100.csv",
        "indicator_code": "EG.IMP.CONS.ZS",
        "params": {
            "format": "json",
            "per_page": 20000,
            "date": "2000:2025",
        },
        "overwrite": True,
        "clip_0_100": True,
    },
    {
        "file_name": "WorldBank_Public_Debt_GDP_2000_2025.csv",
        "indicator_code": "GC.DOD.TOTL.GD.ZS",
        "params": {
            "format": "json",
            "per_page": 20000,
            "date": "2000:2025",
        },
        "overwrite": True,
    },
]


def inspect_eurostat_dimensions(dataset_code: str) -> None:
    """Print the dimension ids and available codes for a Eurostat dataset."""
    url = f"{EUROSTAT_BASE}/{dataset_code}"
    params = {"format": "JSON", "lang": "EN", "lastTimePeriod": 1}
    response = requests.get(url, params=params, headers=EUROSTAT_HEADERS, timeout=60)
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text[:300]}")
        return

    data = response.json()
    print(f"\nDimensions of '{dataset_code}':")
    print(f"  Order: {data['id']}")
    for dim_id in data["id"]:
        categories = data["dimension"][dim_id]["category"]
        labels = categories.get("label", {})
        codes = sorted(categories["index"], key=lambda code: categories["index"][code])
        print(f"\n  [{dim_id}]")
        for code in codes[:20]:
            print(f"    {code:30s} -> {labels.get(code, '')}")
        if len(codes) > 20:
            print(f"    ... ({len(codes) - 20} additional codes)")


def download_eurostat_dataset(dataset: dict) -> None:
    output_path = DATA_RAW_DIR / dataset["file_name"]
    print(f"Processing: {dataset['file_name']}...")

    if output_path.exists():
        print(f"  -> File already exists at '{output_path}'. Skipping download.\n")
        return

    response = requests.get(
        f"{EUROSTAT_BASE}/{dataset['dataset_code']}",
        params=dataset["params"],
        headers=EUROSTAT_HEADERS,
        timeout=120,
    )

    if response.status_code == 400:
        raise RuntimeError(
            f"Eurostat 400 Bad Request.\n"
            f"Detail: {response.text}\n"
            f"Hint: uncomment inspect_eurostat_dimensions() to verify the correct "
            f"dimension names and filter codes for this dataset."
        )
    if response.status_code == 413:
        raise RuntimeError("Eurostat 413: asynchronous response. Wait ~2 minutes and re-run the cell.")
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    ids = data["id"]
    dimensions = data["dimension"]
    sizes = data["size"]
    values = data["value"]

    ordered_codes = []
    for dim_id in ids:
        categories = dimensions[dim_id]["category"]
        ordered_codes.append(sorted(categories["index"], key=lambda code: categories["index"][code]))

    records = []
    for combo in itertools.product(*[range(size) for size in sizes]):
        flat_idx = 0
        stride = 1
        for index in range(len(sizes) - 1, -1, -1):
            flat_idx += combo[index] * stride
            stride *= sizes[index]

        value = values.get(str(flat_idx)) if isinstance(values, dict) else (values[flat_idx] if flat_idx < len(values) else None)
        if value is None:
            continue

        row = {ids[index]: ordered_codes[index][combo[index]] for index in range(len(ids))}
        row["OBS_VALUE"] = float(value)
        records.append(row)

    if not records:
        raise ValueError(f"No data after parsing {dataset['file_name']}.")

    df_raw = pd.DataFrame(records)
    df_tidy = (
        df_raw[dataset["keep_columns"]]
        .rename(columns=dataset["rename_map"])
    )
    df_tidy = convert_eurostat_geo_to_iso3(df_tidy)
    df_tidy = df_tidy.sort_values(["Country", "Year"]).reset_index(drop=True)

    df_tidy.to_csv(output_path, index=False)
    print(f"  -> Success! {len(df_tidy)} rows saved to '{output_path}'.\n")


for dataset in EUROSTAT_DATASETS:
    download_eurostat_dataset(dataset)


def download_world_bank_dataset(dataset: dict) -> None:
    output_path = DATA_RAW_DIR / dataset["file_name"]
    print(f"Processing: {dataset['file_name']}...")

    if output_path.exists() and not dataset.get("overwrite", False):
        print(f"  -> File already exists at '{output_path}'. Skipping download.\n")
        return

    if output_path.exists() and dataset.get("overwrite", False):
        output_path.unlink()

    response = requests.get(
        f"{WORLD_BANK_BASE}/country/all/indicator/{dataset['indicator_code']}",
        params=dataset["params"],
        headers=WORLD_BANK_HEADERS,
        timeout=120,
    )
    response.raise_for_status()

    payload = response.json()
    if not isinstance(payload, list) or len(payload) < 2:
        raise ValueError("Unexpected World Bank response format.")

    valid_country_codes = get_world_bank_country_codes()

    records = []
    for item in payload[1]:
        if not item:
            continue

        country_code = item.get("countryiso3code")
        year_value = item.get("date")
        value = item.get("value")

        if not country_code or country_code not in valid_country_codes:
            continue
        if value is None:
            continue

        year = int(year_value)
        if year < 2000 or year > 2025:
            continue

        value = float(value)
        if dataset.get("clip_0_100", False):
            value = max(0.0, min(100.0, value))

        records.append({
            "Country": country_code,
            "Year": year,
            "Value": value,
        })

    if not records:
        raise ValueError(f"No World Bank public-debt records were returned for {dataset['file_name']}.")

    df_tidy = (
        pd.DataFrame(records)
        .sort_values(["Country", "Year"])
        .reset_index(drop=True)
    )

    df_tidy.to_csv(output_path, index=False)
    print(f"  -> Success! {len(df_tidy)} rows saved to '{output_path}'.\n")


for dataset in WORLD_BANK_DATASETS:
    download_world_bank_dataset(dataset)


def build_global_public_debt() -> None:
    """Build a single global public-debt series preferring Eurostat values when available.

    Rationale: the project keeps the Eurostat series as authoritative where available
    (EU and covered countries) and uses World Bank values only where Eurostat is not available.
    This function reads the World Bank and Eurostat raw CSVs (when present), concatenates them
    with a source-priority marker and drops duplicate Country-Year keeping the
    higher-priority source (Eurostat).
    """

    world_bank_path = DATA_RAW_DIR / "WorldBank_Public_Debt_GDP_2000_2025.csv"
    eurostat_path = DATA_RAW_DIR / "Eurostat_Public_Debt_GDP_2000_2025.csv"
    output_paths = DATA_RAW_DIR / "Global_Public_Debt_GDP_2000_2025.csv"

    # Require at least one source
    if not world_bank_path.exists() and not eurostat_path.exists():
        raise FileNotFoundError(
            f"Missing both World Bank and Eurostat public debt files: {world_bank_path}, {eurostat_path}"
        )

    combined_frames = []

    # Prefer Eurostat values: assign lower priority number (0) to Eurostat so they are
    # kept when dropping duplicates. Use World Bank only where Eurostat is missing.
    if eurostat_path.exists():
        eurostat_df = pd.read_csv(eurostat_path)
        combined_frames.append(eurostat_df.assign(_source_priority=0))

    if world_bank_path.exists():
        world_bank_df = pd.read_csv(world_bank_path)
        combined_frames.append(world_bank_df.assign(_source_priority=1))

    merged = (
        pd.concat(combined_frames, ignore_index=True)
        .sort_values(["Country", "Year", "_source_priority"])
        .drop_duplicates(["Country", "Year"], keep="first")
        .drop(columns=["_source_priority"])
        .sort_values(["Country", "Year"])
        .reset_index(drop=True)
    )

    merged.to_csv(output_paths, index=False)
    print(f"  -> Success! {len(merged)} rows saved to '{output_paths}'. (Eurostat preferred where present)\n")



build_global_public_debt()

Processing: Eurostat_Public_Debt_GDP_2000_2025.csv...
  -> File already exists at 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\Eurostat_Public_Debt_GDP_2000_2025.csv'. Skipping download.

Processing: Global_Energy_Dependency_0_100.csv...
  -> Success! 3151 rows saved to 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\Global_Energy_Dependency_0_100.csv'.

Processing: WorldBank_Public_Debt_GDP_2000_2025.csv...
  -> Success! 1131 rows saved to 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\WorldBank_Public_Debt_GDP_2000_2025.csv'.

  -> Success! 1856 rows saved to 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\Global_Public_Debt_GDP_2000_2025.csv'. (Eurostat preferred where present)

